# encrypted password vault using AES encryption

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install cryptography

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import padding, hashes
from cryptography.hazmat.backends import default_backend
from getpass import getpass
import base64, os, json

vault_file = "vault.json"
backend = default_backend()
salt = b'\x9a\xd2\x19\x04\x8f\x18\x1f\x91'

def derive_key(password):
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=100000, backend=backend)
    return kdf.derive(password.encode())

def encrypt(data, key):
    iv = os.urandom(16)
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=backend)
    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(data.encode()) + padder.finalize()
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(padded_data) + encryptor.finalize()
    return base64.b64encode(iv + ciphertext).decode()

def decrypt(token, key):
    data = base64.b64decode(token.encode())
    iv, ciphertext = data[:16], data[16:]
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=backend)
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    return unpadder.update(padded_data) + unpadder.finalize()

def load_vault():
    if not os.path.exists(vault_file):
        return {}
    with open(vault_file, "r") as f:
        return json.load(f)

def save_vault(data):
    with open(vault_file, "w") as f:
        json.dump(data, f)

def main():
    master = getpass("Master password: ")
    key = derive_key(master)
    vault = load_vault()
    while True:
        choice = input("\n1. Add\n2. View\n3. Delete\n4. Exit\n> ")
        if choice == "1":
            site = input("Site: ")
            user = input("Username: ")
            pwd = getpass("Password: ")
            entry = json.dumps({"username": user, "password": pwd})
            vault[site] = encrypt(entry, key)
            save_vault(vault)
        elif choice == "2":
            site = input("Site to view: ")
            if site in vault:
                try:
                    data = json.loads(decrypt(vault[site], key).decode())
                    print(f"Username: {data['username']}\nPassword: {data['password']}")
                except:
                    print("Decryption failed.")
            else:
                print("Not found.")
        elif choice == "3":
            site = input("Site to delete: ")
            if site in vault:
                del vault[site]
                save_vault(vault)
                print("Deleted.")
            else:
                print("Not found.")
        elif choice == "4":
            break
        else:
            print("Invalid.")

main()
